In [1]:
from pathlib import Path
import json
import pandas as pd

BASE_DIR = Path(".").resolve()
DA_MAPPINGS_DIR = BASE_DIR / "da_mappings" / "da_mappings_cox"


In [2]:
from IPython.display import display, HTML

rows = []

for fp in sorted(DA_MAPPINGS_DIR.glob("*.json")):
    data = json.loads(fp.read_text(encoding="utf-8"))
    firm_id = fp.stem

    for var_block in data.get("variables", []):
        variable = var_block.get("variable")
        final_choice = var_block.get("final_choice", [])

        if final_choice:  # only keep non-empty final_choice
            for ch in final_choice:
                rows.append({
                    "firm_id": firm_id,
                    "variable": variable,
                    "sheet_name": ch.get("sheet_name", ""),
                    "row_label": ch.get("row_label", ""),
                    "needs_manual_review": var_block.get("needs_manual_review", False),
                    "notes": var_block.get("notes", ""),
                })

df_nonempty = pd.DataFrame(rows)

if df_nonempty.empty:
    print("No non-empty final_choice found.")
else:
    df_show = df_nonempty.sort_values([ "firm_id","variable"]).reset_index(drop=True)

    html_table = df_show.to_html(index=False, escape=False)

    display(HTML(f"""
    <div style="
        max-height: 600px;
        overflow-y: auto;
        overflow-x: auto;
        border: 1px solid #ccc;
        padding: 6px;
        background: black;
    ">
        {html_table}
    </div>
    """))

firm_id,variable,sheet_name,row_label,needs_manual_review,notes
AAB.CO,XSGA_DA,Income Statement,Dep/Office Fixtures/Premises,False,"The selected XSGA bucket contains overhead expense rows (Selling, General & Admin; Administrative Exp.; Other External Expenses; Personnel Expenses) and explicitly excluded depreciation/amortization in the earlier step. These three depreciation rows appear as separate same-level operating lines above EBIT, so they should be added separately later. 'Amortization of Intangibles' is excluded because it appears linked to transfer/intangible player-rights activity rather than SG&A."
AAB.CO,XSGA_DA,Income Statement,Depreciation of Equipment and Inventories,False,"The selected XSGA bucket contains overhead expense rows (Selling, General & Admin; Administrative Exp.; Other External Expenses; Personnel Expenses) and explicitly excluded depreciation/amortization in the earlier step. These three depreciation rows appear as separate same-level operating lines above EBIT, so they should be added separately later. 'Amortization of Intangibles' is excluded because it appears linked to transfer/intangible player-rights activity rather than SG&A."
AAB.CO,XSGA_DA,Income Statement,Depreciation of Financial Lease Right-of-Use Assets,False,"The selected XSGA bucket contains overhead expense rows (Selling, General & Admin; Administrative Exp.; Other External Expenses; Personnel Expenses) and explicitly excluded depreciation/amortization in the earlier step. These three depreciation rows appear as separate same-level operating lines above EBIT, so they should be added separately later. 'Amortization of Intangibles' is excluded because it appears linked to transfer/intangible player-rights activity rather than SG&A."
AGFEb.CO,XSGA_DA,Income Statement,Depreciation/Amortization,False,"A standalone operating 'Depreciation/Amortization' line is shown separately from 'Selling and Marketing Expenses', 'Administrative Expenses', 'Personnel Expenses', and 'Operating Expenses'. Relative to the selected XSGA_COMPONENTS bucket, this suggests D&A is outside the previously selected rows and should be added separately. The summary row is chosen instead of individual depreciation/amortization components."
AOJb.CO,XSGA_DA,Income Statement,Depreciation and amortisation,False,"Use the summary D&A row only. Do not also select the asset-type breakdown rows ('Lease Assets, External', 'Intangible assets', 'Property, plant and equipment') to avoid double counting."
BETCO.CO,XSGA_DA,Income Statement,Depreciation,False,"Selected XSGA bucket consists only of 'Staff Costs' and 'Other external expenses'. The D&A rows appear as separate same-level operating expenses, so they should be added separately to avoid understating operating overhead. The blank 'Depreciation / Amortization' row is not used because it appears to be a header, not a numeric summary."
BETCO.CO,XSGA_DA,Income Statement,Amortization and impairment,False,"Selected XSGA bucket consists only of 'Staff Costs' and 'Other external expenses'. The D&A rows appear as separate same-level operating expenses, so they should be added separately to avoid understating operating overhead. The blank 'Depreciation / Amortization' row is not used because it appears to be a header, not a numeric summary."
BIF.CO,XSGA_DA,Income Statement,Depreciation,False,"Selected SG&A reference bucket consists of 'Selling/Advertising', 'Administrative', and 'Staff Costs'. The visible D&A lines are presented separately at the operating level rather than as sub-components of those selected rows, so they should be added separately. The impairment-containing summary row was not chosen to avoid including impairment. 'Amortization of Contract Rights' appears transfer-related/non-SG&A and is excluded."
BIF.CO,XSGA_DA,Income Statement,Amortization of Other Rights,False,"Selected SG&A reference bucket consists of 'Selling/Advertising', 'Administrative', and 'Staff Costs'. The visible D&A lines are presented separately at th

In [3]:
df_manual = df_nonempty[df_nonempty["needs_manual_review"] == True]

print(f"Number of rows:                    {len(df_nonempty)}")
print(f"Unique tickers (firm_id):          {df_nonempty['firm_id'].nunique()}")
print(f"\nNeeds manual review (total rows):  {len(df_manual)}")
print(f"Needs manual review (unique tickers): {df_manual['firm_id'].nunique()}")
print(f"\nAll tickers: {sorted(df_nonempty['firm_id'].unique().tolist())}")

Number of rows:                    73
Unique tickers (firm_id):          40

Needs manual review (total rows):  3
Needs manual review (unique tickers): 2

All tickers: ['AAB.CO', 'AGFEb.CO', 'AOJb.CO', 'BETCO.CO', 'BIF.CO', 'BOOZT.CO', 'CBRAIN.CO', 'CEMAT.CO', 'CHEMM.CO', 'COLUM.CO', 'CPHCAPST.CO', 'DFDS.CO', 'DNORD.CO', 'DSV.CO', 'ENSG.CO', 'FFARMS.CO', 'GABR.CO', 'GREENM.CO', 'GYLDb.CO', 'HUSCO.CO', 'KLEEb.CO', 'MATAS.CO', 'NETCG.CO', 'NLFSK.CO', 'NORTHM.CO', 'NTGNT.CO', 'OKEAC.CO', 'PAALb.CO', 'PARKEN.CO', 'PARKSTa.CO', 'PFINV.CO', 'RIASb.CO', 'ROVS.CO', 'RTX.CO', 'SIGR.CO', 'SKAKO.CO', 'SOLARb.CO', 'TCM.CO', 'TRIFOR.CO', 'TRMDa.CO']
